<a href="https://colab.research.google.com/github/SharmisthaDey-lab/python-demos/blob/main/Project_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Web Scrapping : From HTML to Structured Data**

In this notebook, I have extracting infomrtaion from the web. We will use the source [Books to Scrape](https://www.google.com/url?q=http%3A%2F%2Fbooks.toscrape.com%2F) sandbox to build a tool that extracts "titles", "prices" and "ratings" into clean dataset

##**Step 1. Send HTTP request to website**



In [32]:
import requests #Request librrary for connection with the website.

#Target sandbox URL
url = "http://books.toscrape.com/index.html"

#Send a GET request
response = requests.get(url)

#Check status of get request
print(f"Connection status: {response.status_code}")

#Visual of raw html data received via get request- fist 500 characters
print(f"\nRaw HTML preview:,{response.text[:500]}")


Connection status: 200

Raw HTML preview:,<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


##**Step 2. Understand HTML structure**



In [33]:
from bs4 import BeautifulSoup

#First turn messy HTML text into python object
soup = BeautifulSoup(response.text,"lxml")

#Find all the books on the page and store in books list - These are specific tags from HTML
books = soup.find_all('article',class_="product_pod")

print(f"Number of books found: {len(books)}")

Number of books found: 20


##**Step 3. Extract specific meta infomrtaion for each book for one sample Book for confirmation**

In [34]:
#Test extraction for first book
sample_book = books[0]
title = sample_book.find('h3').find('a')['title']
price = sample_book.find('p',class_="price_color").text.strip()
rating = sample_book.find('p',class_='star-rating')['class'][1]
stock = sample_book.find('p', class_='instock availability').text.strip() #Stripe() removes white spaces

print(f"Example scrap: \n\nTitle: {title}\nPrice: {price}\nRating: {rating}\nStock: {stock}")

Example scrap: 

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Stock: In stock


##**Step 4. Implement scrapper function to extract all books and its meta information into a tabular output**

In [35]:
import pandas as pd

def scrape_book_details(url):
  response = requests.get(url)
  soup = BeautifulSoup(response.text,"lxml")
  books_container = soup.find_all('article',class_="product_pod")

  scrapped_data = []
  for book in books_container:
    try:
      details = {
        "Titles": book.find('h3').find('a')['title'],
        "Prices": book.find('p',class_="price_color").text.strip(),
        "Ratings": book.find('p',class_='star-rating')['class'][1],
        "Stock": book.find('p', class_='instock availability').text.strip()
      }
      scrapped_data.append(details)
    except Exception as e:
      print(f"Log Error: {e}")
      continue

  return scrapped_data


results = scrape_book_details(url)

data_frame = pd.DataFrame(results) #Pandas dataframe function to dispay tabular results
data_frame.head() #Display only top few rows


,Titles,Prices,Ratings,Stock
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock


##**Step 5. Download into .csv**

In [36]:
data_frame.to_csv("scrapped_data.csv")


## **5. Reflection & Next Steps**

* **Cleaning:** Notice the price contains a "Â" character? That's an encoding artifact. How would you use `.replace()` to clean that?
* **Data Types:** The rating is a string (e.g., "Three"). For analysis, we should map these to numbers (1–5).
* **Pagination:** This page only has 20 books. Real scrapers look for the "Next" button URL and loop until every page is finished.